# Streams, Tasks & Dynamic Tables

## Test Cases
| Feature | Test Scenario | Target |
|---------|--------------|--------|
| Streams/CDC | Create stream, capture 10K changes, consume via Task | GA |
| Dynamic Tables | Create DT with 1-min lag from Iceberg source | Verify incremental refresh |
| Snowpipe Streaming | Stream 100K JSON records | P99 latency <10s |

In [ ]:
-- Set context
USE ROLE ACCOUNTADMIN;
USE DATABASE ICEBERG_POC;
USE SCHEMA TESTS;
USE WAREHOUSE COMPUTE_WH;

## Test 1: Streams & CDC
Create stream on Iceberg table and capture changes.

In [ ]:
-- Create CLAIMS_STAGING as the CDC source (new claim submissions arrive here)
CREATE OR REPLACE ICEBERG TABLE ICEBERG_POC.TESTS.CLAIMS_STAGING (
    claim_id        INT,
    encounter_id    INT,
    patient_id      INT,
    payer_name      VARCHAR,
    claim_status    VARCHAR,
    submitted_date  DATE,
    billed_amount   DECIMAL(10,2)
)
CATALOG = SNOWFLAKE
EXTERNAL_VOLUME = EXVOL
ICEBERG_VERSION = 3;

-- Create append-only stream on CLAIMS_STAGING
CREATE OR REPLACE STREAM ICEBERG_POC.TESTS.CLAIMS_STREAM
  ON TABLE ICEBERG_POC.TESTS.CLAIMS_STAGING APPEND_ONLY = TRUE;

-- Create CLAIMS_PROCESSED target table
CREATE OR REPLACE TABLE ICEBERG_POC.TESTS.CLAIMS_PROCESSED (
    claim_id        INT,
    encounter_id    INT,
    patient_id      INT,
    payer_name      VARCHAR,
    claim_status    VARCHAR,
    submitted_date  DATE,
    billed_amount   DECIMAL(10,2),
    change_type     VARCHAR,
    captured_at     TIMESTAMP_NTZ
);

In [ ]:
-- Create Task to consume CLAIMS_STREAM and write to CLAIMS_PROCESSED
CREATE OR REPLACE TASK ICEBERG_POC.TESTS.CLAIMS_INTAKE_TASK
    WAREHOUSE = COMPUTE_WH
    SCHEDULE = '1 MINUTE'
    WHEN SYSTEM$STREAM_HAS_DATA('ICEBERG_POC.TESTS.CLAIMS_STREAM')
AS
INSERT INTO ICEBERG_POC.TESTS.CLAIMS_PROCESSED
SELECT
    claim_id, encounter_id, patient_id, payer_name, claim_status, submitted_date, billed_amount,
    CASE
        WHEN METADATA$ACTION = 'INSERT' THEN 'INSERT'
        WHEN METADATA$ACTION = 'DELETE' AND METADATA$ISUPDATE THEN 'UPDATE'
        ELSE 'DELETE'
    END AS change_type,
    CURRENT_TIMESTAMP() AS captured_at
FROM ICEBERG_POC.TESTS.CLAIMS_STREAM;

-- Resume task
ALTER TASK ICEBERG_POC.TESTS.CLAIMS_INTAKE_TASK RESUME;

In [ ]:
-- Insert new claim submissions to trigger CDC stream
INSERT INTO ICEBERG_POC.TESTS.CLAIMS_STAGING
SELECT
    SEQ4() + 9000001    AS claim_id,
    SEQ4() % 1000000 + 3001 AS encounter_id,
    SEQ4() % 100000  + 1001 AS patient_id,
    ARRAY_CONSTRUCT('Blue Cross','Aetna','United Healthcare','Cigna','Humana')[SEQ4() % 5]::VARCHAR AS payer_name,
    'Submitted'          AS claim_status,
    CURRENT_DATE()       AS submitted_date,
    ROUND((SEQ4() % 9000 + 500) / 10.0, 2) AS billed_amount
FROM TABLE(GENERATOR(ROWCOUNT => 10000));

-- Check stream has data
SELECT SYSTEM$STREAM_HAS_DATA('ICEBERG_POC.TESTS.CLAIMS_STREAM') AS has_data;
SELECT COUNT(*) AS stream_rows FROM ICEBERG_POC.TESTS.CLAIMS_STREAM;

## Test 2: Dynamic Tables
Create Dynamic Table with 1-min lag from Iceberg source.

In [ ]:
-- Create ENCOUNTER_SUMMARY Dynamic Table from ENCOUNTERS + PATIENTS Iceberg source
CREATE OR REPLACE DYNAMIC TABLE ICEBERG_POC.TESTS.ENCOUNTER_SUMMARY
    TARGET_LAG = '1 minute'
    WAREHOUSE = COMPUTE_WH
    REFRESH_MODE = INCREMENTAL
AS
SELECT
    e.encounter_type,
    p.state,
    DATE_TRUNC('month', e.encounter_date) AS encounter_month,
    COUNT(*)                               AS encounter_count,
    ROUND(SUM(e.total_charge), 2)          AS total_charge,
    ROUND(AVG(e.total_charge), 2)          AS avg_charge,
    COUNT(DISTINCT e.patient_id)           AS unique_patients
FROM ICEBERG_POC.TESTS.ENCOUNTERS e
JOIN ICEBERG_POC.TESTS.PATIENTS   p ON e.patient_id = p.patient_id
GROUP BY e.encounter_type, p.state, DATE_TRUNC('month', e.encounter_date);

-- Check Dynamic Table status
SHOW DYNAMIC TABLES LIKE 'ENCOUNTER_SUMMARY' IN SCHEMA ICEBERG_POC.TESTS;

In [ ]:
-- Insert new encounters to verify ENCOUNTER_SUMMARY incremental refresh
INSERT INTO ICEBERG_POC.TESTS.ENCOUNTERS
SELECT
    (SELECT MAX(encounter_id) FROM ICEBERG_POC.TESTS.ENCOUNTERS) + SEQ4() AS encounter_id,
    SEQ4() % 100000 + 1001  AS patient_id,
    SEQ4() % 1000   + 2001  AS provider_id,
    CURRENT_DATE()           AS encounter_date,
    'Preventive Care'        AS encounter_type,
    'Z00.00'                 AS primary_diagnosis_code,
    'Annual wellness visit'  AS primary_diagnosis_desc,
    'Discharged'             AS disposition,
    ROUND((SEQ4() % 2000 + 100) / 10.0, 2) AS total_charge
FROM TABLE(GENERATOR(ROWCOUNT => 100000));

-- Check Dynamic Table incremental refresh result
SELECT * FROM ICEBERG_POC.TESTS.ENCOUNTER_SUMMARY
WHERE encounter_type = 'Preventive Care'
ORDER BY encounter_month DESC
LIMIT 5;